In [ ]:
# ==============================================================
# Install the required Python libraries
#
# pandas       : Data loading, preprocessing, and CSV file handling.
# NumPy        : Numerical computing and array operations.
# Matplotlib   : Visualization of training, validation, and prediction results.
# TensorFlow   : Deep learning framework for building and training the
#                Liquid Time-Constant (LTC) neural network.
# keras-ncp    : Implements the Neural Circuit Policies (NCP) architecture,
#                including the LTCCell used in the proposed LTC model.
# XlsxWriter   : Exporting results and performance metrics to Excel files.
#
# Run this cell only once when executing the notebook in a new Python
# environment.
# ==============================================================

!pip install pandas numpy matplotlib
!pip install tensorflow
!pip install keras-ncp
!pip install xlsxwriter


In [ ]:
# ==============================================================
# Import required libraries
# ==============================================================
import tensorflow as tf

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
import copy
import csv
import math
import cmath
import time
import random as python_random

# ==============================================================
# Set random seeds to ensure reproducible training results
# ==============================================================
def reset_seeds():
    np.random.seed(7)
    python_random.seed(7)
    tf.random.set_seed(7)

# ==============================================================
# Load the source-domain dataset (Pretraining Dataset)
# Input_1.csv  : Input features
# Output_1.csv : Target output
# ==============================================================
inFilename = "Input_1.csv"
outFilename = "Output_1.csv"

input = pd.read_csv(inFilename, header=None)
output = pd.read_csv(outFilename, header=None)

inputRawTrain = np.array(input)
outputRawTrain = np.array(output)

# ==============================================================
# Load the target-domain dataset (Transfer Learning Dataset)
# Input_2.csv  : Input features
# Output_2.csv : Target output
# ==============================================================
inFilename = "Input_2.csv"
outFilename = "Output_2.csv"

input = pd.read_csv(inFilename, header=None)
output = pd.read_csv(outFilename, header=None)

inputTrain_1 = np.array(input)
outputTrain_1 = np.array(output)

# ==============================================================
# Display the total number of samples in the target dataset
# ==============================================================
num_inputs_train = len(inputTrain_1)
print("Total Number of Training Dataset is:", num_inputs_train)

# ==============================================================
# Randomly shuffle the dataset while preserving the correspondence
# between the input features and output targets
# ==============================================================
np.random.seed(1)
np.random.shuffle(inputTrain_1)

np.random.seed(1)
np.random.shuffle(outputTrain_1)

# ==============================================================
# Split the target dataset into:
#   - 70% Training set
#   - 15% Validation set
#   - 15% Testing set
# ==============================================================
TRAIN_SPLIT_1 = int(0.70 * num_inputs_train)
TRAIN_SPLIT_2 = int(0.85 * num_inputs_train)

inputTrain, inputTest, inputs_test_2 = np.split(
    inputTrain_1, [TRAIN_SPLIT_1, TRAIN_SPLIT_2]
)

outputTrain, outputTest, outputs_test_2 = np.split(
    outputTrain_1, [TRAIN_SPLIT_1, TRAIN_SPLIT_2]
)

print("Dataset preparation complete!")

In [ ]:
# ==============================================================
# Define the number of independent randomization cycles
# Each cycle generates a different shuffled version of the source
# and target training datasets for robust model evaluation.
# ==============================================================
CYCLE = 5

# ==============================================================
# Determine the number of samples in each dataset
# ==============================================================
num_inputRawTrain = len(inputRawTrain)
num_inputTrain = len(inputTrain)
num_inputTest = len(inputTest)

# ==============================================================
# Assign the testing dataset for model evaluation
# ==============================================================
inputs_test = inputTest
outputs_test = outputTest

print("Total Number of Raw Training Dataset is:", num_inputRawTrain)
print("Total Number of Real Training Dataset is:", num_inputTrain)
print("Total Number of Test Dataset is:", num_inputTest)

# ==============================================================
# Initialize lists to store multiple randomized versions of the
# source and target training datasets
# ==============================================================
inputs_train_raw = []
outputs_train_raw = []
inputs_train_real = []
outputs_train_real = []

# ==============================================================
# Generate multiple randomized training datasets
#
# For each cycle:
#   1. Create independent copies of the datasets.
#   2. Shuffle the source-domain dataset.
#   3. Shuffle the target-domain dataset.
#   4. Store the randomized datasets for subsequent training.
#
# The same random seed is applied to the input and output arrays
# to preserve the correspondence between samples and labels.
# ==============================================================
for x in range(CYCLE):

    # Create deep copies of the datasets
    tempInput_raw = copy.deepcopy(inputRawTrain)
    tempOutput_raw = copy.deepcopy(outputRawTrain)

    tempInput_real = copy.deepcopy(inputTrain)
    tempOutput_real = copy.deepcopy(outputTrain)

    # Randomize the source-domain (pretraining) dataset
    np.random.seed(x + 1)
    np.random.shuffle(tempInput_raw)

    np.random.seed(x + 1)
    np.random.shuffle(tempOutput_raw)

    inputs_train_raw.append(tempInput_raw)
    outputs_train_raw.append(tempOutput_raw)

    # Randomize the target-domain (transfer learning) dataset
    np.random.seed(x + 1)
    np.random.shuffle(tempInput_real)

    np.random.seed(x + 1)
    np.random.shuffle(tempOutput_real)

    inputs_train_real.append(tempInput_real)
    outputs_train_real.append(tempOutput_real)

print("Dataset randomization and separation complete!")

In [ ]:
# ==============================================================
# This code implements the Liquid Time-Constant (LTC) neural network
# with transfer learning (TL) to predict the dynamic behavior of a
# modified CIGRE active distribution network (ADN) integrated with
# the IEEE 9-bus transmission system (TS).
#
# The complete implementation is organized into the following phases.
# (Phase 0: Pretraining, Phase 1: Initialization,
#  Phase 2: Warm-up, Phase 3: Progressive fine-tuning)
#
# Compatible with your SCRATCH variable names:
#   inputs_train_raw, outputs_train_raw, num_inputRawTrain
#   inputs_train_real, outputs_train_real   (lists of 5 splits)
#   inputs_test, outputs_test
#
# POWER setup:
#   Inputs : 5 features [F, V, Ps, Pw, Pb]
#   Output : 1 feature (P or Q)
#
# Algorithm 2 Implementation
# - Phase 1 (Initialization): mu_t, sigma_t^2 explicitly re-adapted on
#   Dt right after Mt <- Ms (this was already happening implicitly;
#   now it's pulled out and labeled to match lines 6-8 of the algorithm).
# - Phase 2 (Warm-up): now freezes EVERYTHING except the final output
#   layer (previously it froze only the LTC cell and left the whole
#   head trainable). This is what makes Phase 3's progressive
#   unfreezing meaningful -- otherwise there is nothing left to
#   progressively unfreeze.
# - Phase 3 (Progressive fine-tuning): NEW. Implements the
#   k = 1..N loop from the algorithm:
#       E_k = T_w + floor(k*(T - T_w)/N)
#       at stage k, unfreeze layer (L-k+1) counting from the output,
#       previously-unfrozen layers stay trainable,
#       train from epoch E_{k-1}+1 to E_k.
#   Layer order (input -> output), L = 5:
#       1: ltc   2: head_dense_1   3: head_skip
#       4: head_dense_2   5: power_output
#   So stage k=1 keeps/unfreezes power_output, k=2 adds head_dense_2,
#   k=3 adds head_skip, k=4 adds head_dense_1, k=5 adds ltc (full FT).
#   N and T (total progressive-phase epoch budget) are both
#   configurable constants below.
#
# Requires:
#   pip install keras-ncp
# ==============================================================

import os, math, random
import numpy as np
import tensorflow as tf
from kerasncp.wirings import NCP
from kerasncp.tf import LTCCell
from tensorflow.keras import regularizers

# -------------------------
# Reproducibility
# -------------------------
def reset_seeds(seed: int = 7):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

reset_seeds(7)

# -------------------------
# Helpers: robust "maybe list" extractor
# -------------------------
def _first_if_list(x):
    return x[0] if isinstance(x, (list, tuple)) else x

# -----------------------
# Normalization & scaling
# -----------------------
def create_normalization_layer(data):
    norm = tf.keras.layers.Normalization()
    norm.adapt(np.asarray(data, dtype=np.float32))
    return norm

class OutputStandardizer:
    """Z-score scaler (reversible) for POWER (1D)."""
    def __init__(self, y_train, eps: float = 1e-12):
        y = np.asarray(y_train, dtype=np.float32).reshape(-1, 1)
        self.mean = y.mean(axis=0, keepdims=True)
        self.std  = y.std(axis=0, keepdims=True)
        self.std  = np.where(self.std < eps, eps, self.std)

    def transform(self, y):
        return (np.asarray(y, dtype=np.float32).reshape(-1, 1) - self.mean) / self.std

    def inverse(self, y_hat):
        return np.asarray(y_hat, dtype=np.float32).reshape(-1, 1) * self.std + self.mean

# --------------------------------------
# LTC temporal depth: repeat static input
# --------------------------------------
def _tile_static_to_steps(x, steps: int):
    x = tf.expand_dims(x, axis=1)
    return tf.tile(x, [1, steps, 1])

# --------------------------------------
# Build LTC POWER body: (5)->(1)
# --------------------------------------
def build_LTC_power_body(
    n_features: int = 5,
    inter_neurons=32, command_neurons=16, motor_neurons=1,
    head_width=128,
    l2w=0.0, drop=0.0, gaussian_noise_std=0.0,
    ltc_steps: int = 8,
):
    wiring = NCP(
        inter_neurons=int(inter_neurons),
        command_neurons=int(command_neurons),
        motor_neurons=int(motor_neurons),
        sensory_fanout=4, inter_fanout=4,
        recurrent_command_synapses=4, motor_fanin=4
    )
    ltc_cell = LTCCell(wiring)

    inp = tf.keras.Input(shape=(n_features,), dtype=tf.float32, name="features_normed")
    x = tf.keras.layers.GaussianNoise(gaussian_noise_std, name="gn_input")(inp)
    x = tf.keras.layers.Lambda(lambda t: _tile_static_to_steps(t, ltc_steps), name="repeat_T")(x)
    h = tf.keras.layers.RNN(ltc_cell, return_sequences=False, name="ltc")(x)

    H = int(head_width)
    reg = regularizers.l2(l2w) if (l2w and l2w > 0) else None

    h1 = tf.keras.layers.Dense(H, activation="gelu", kernel_regularizer=reg, name="head_dense_1")(h)
    h1 = tf.keras.layers.Dropout(drop, name="head_dropout")(h1)
    h2 = tf.keras.layers.Dense(H, activation="gelu", kernel_regularizer=reg, name="head_dense_2")(h1)
    skip = tf.keras.layers.Dense(H, activation=None, kernel_regularizer=reg, name="head_skip")(h)

    z = tf.keras.layers.Add(name="head_add")([h2, skip])
    z = tf.keras.layers.LayerNormalization(name="head_norm")(z)
    out = tf.keras.layers.Dense(1, name="power_output", kernel_regularizer=reg)(z)

    return tf.keras.Model(inp, out, name="LTC_power_body")

# -----------------------------------------------------------
# Layer bookkeeping for progressive unfreezing (Algorithm 2)
# -----------------------------------------------------------
def _walk_layers(layer_or_model):
    yield layer_or_model
    if hasattr(layer_or_model, "layers") and layer_or_model.layers:
        for sub in layer_or_model.layers:
            yield from _walk_layers(sub)

# Ordered input -> output. This defines "L" and the identity of
# theta_{L-k+1} in Algorithm 2's Phase 3 loop.
FT_LAYER_ORDER = ["ltc", "head_dense_1", "head_skip", "head_dense_2", "power_output"]
L_TOTAL_LAYERS = len(FT_LAYER_ORDER)

def _find_named_layer(model, name):
    for layer in _walk_layers(model):
        if getattr(layer, "name", None) == name:
            return layer
    return None

def freeze_all_ft_layers(model):
    """theta_l frozen for all l (used at the very start of Phase 2)."""
    for name in FT_LAYER_ORDER:
        layer = _find_named_layer(model, name)
        if layer is not None:
            layer.trainable = False

def set_layer_trainable(model, name, trainable):
    layer = _find_named_layer(model, name)
    if layer is None:
        raise ValueError(f"Layer '{name}' not found in model.")
    layer.trainable = trainable

def unfreeze_from_output(model, num_unfrozen: int):
    """
    Unfreeze the last `num_unfrozen` layers counting from the OUTPUT
    side, per FT_LAYER_ORDER (which is input->output). Previously
    unfrozen layers stay trainable (cumulative), matching:
        "Unfreeze layer theta_{L-k+1} (previously unfrozen layers
         remain trainable)"
    num_unfrozen == k at stage k.
    """
    num_unfrozen = max(0, min(num_unfrozen, L_TOTAL_LAYERS))
    # Layers closest to the output are at the END of FT_LAYER_ORDER.
    to_unfreeze = FT_LAYER_ORDER[L_TOTAL_LAYERS - num_unfrozen:]
    for name in FT_LAYER_ORDER:
        layer = _find_named_layer(model, name)
        if layer is not None:
            layer.trainable = (name in to_unfreeze)

def trainable_report(model):
    return {name: (_find_named_layer(model, name).trainable
                   if _find_named_layer(model, name) is not None else None)
            for name in FT_LAYER_ORDER}

# --------------------------------------------------------
# Callback: checkpoint by ORIGINAL-SCALE val_loss + history
# --------------------------------------------------------
class OrigScaleMSECheckpoint(tf.keras.callbacks.Callback):
    def __init__(self, filepath):
        super().__init__()
        self.filepath = filepath
        self.best = np.inf
        self.mse_orig_hist = []
        self.val_mse_orig_hist = []

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        mse_tr  = float(logs.get("loss", np.nan))
        mse_val = float(logs.get("val_loss", np.inf))

        logs["mse_orig"] = mse_tr
        logs["val_mse_orig"] = mse_val

        self.mse_orig_hist.append(mse_tr)
        self.val_mse_orig_hist.append(mse_val)

        if mse_val < self.best:
            self.best = mse_val
            self.model.save_weights(self.filepath)
            print(f"\n[Checkpoint] Epoch {epoch+1}: val_MSE_orig = {mse_val:.6e}  -> saved.")

# -----------------------------
# Optimizer + compile
# -----------------------------
def make_optimizer(lr_schedule, use_adamw=False, weight_decay=0.0, clipnorm=1.0):
    if use_adamw and weight_decay > 0:
        try:
            return tf.keras.optimizers.AdamW(
                learning_rate=lr_schedule, weight_decay=weight_decay, clipnorm=clipnorm
            )
        except Exception:
            return tf.keras.optimizers.Adam(learning_rate=lr_schedule, clipnorm=clipnorm)
    return tf.keras.optimizers.Adam(learning_rate=lr_schedule, clipnorm=clipnorm)

def compile_power_model_original_mse(model, optimizer, y_std_np, loss_name="mse", huber_delta=0.01):
    y_std_tf = tf.constant(np.asarray(y_std_np, dtype=np.float32).reshape(1, 1))

    def mse_original(y_true, y_pred):
        err_orig = (y_pred - y_true) * y_std_tf
        return tf.reduce_mean(tf.square(err_orig))

    if loss_name == "huber":
        delta_tf = tf.constant(float(huber_delta), dtype=tf.float32)
        def huber_original(y_true, y_pred):
            err_orig = (y_pred - y_true) * y_std_tf
            abs_e = tf.abs(err_orig)
            quad = tf.minimum(abs_e, delta_tf)
            lin  = abs_e - quad
            return tf.reduce_mean(0.5 * tf.square(quad) + delta_tf * lin)
        loss_fn = huber_original
    else:
        loss_fn = mse_original

    model.compile(optimizer=optimizer, loss=loss_fn, metrics=[loss_fn])

# -----------------------------
# Train one stage (returns model, history4)
# num_epochs here is the number of NEW epochs to run in this stage
# (i.e. E_k - E_{k-1}), NOT a fresh optimizer restart in the sense
# of resetting weights -- weights persist across stages, only the
# LR schedule and trainable mask change per stage.
# -----------------------------
def train_stage(
    model,
    Xtr, Ytr, Xval, Yval,
    y_scaler: OutputStandardizer,
    lr_init, decay_rate, decay_epochs,
    epochs, batch_size,
    ckpt_path,
    use_adamw=False, weight_decay=0.0, clipnorm=1.0,
    loss_name="mse", huber_delta=0.01,
):
    Xtr = np.asarray(Xtr, dtype=np.float32)
    Ytr = np.asarray(Ytr, dtype=np.float32).reshape(-1, 1)
    Xval = np.asarray(Xval, dtype=np.float32)
    Yval = np.asarray(Yval, dtype=np.float32).reshape(-1, 1)

    Ytr_n  = y_scaler.transform(Ytr)
    Yval_n = y_scaler.transform(Yval)

    if epochs <= 0:
        return {"loss": [], "val_loss": [], "mse_orig": [], "val_mse_orig": []}

    steps_per_epoch = max(1, math.ceil(len(Xtr) / batch_size))
    decay_steps = int(steps_per_epoch * int(decay_epochs))

    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=float(lr_init),
        decay_steps=decay_steps,
        decay_rate=float(decay_rate),
        staircase=True,
    )
    opt = make_optimizer(lr_schedule, use_adamw=use_adamw, weight_decay=weight_decay, clipnorm=clipnorm)

    compile_power_model_original_mse(
        model,
        optimizer=opt,
        y_std_np=y_scaler.std,
        loss_name=loss_name,
        huber_delta=huber_delta
    )

    ckpt_cb = OrigScaleMSECheckpoint(ckpt_path)

    hist = model.fit(
        Xtr, Ytr_n,
        validation_data=(Xval, Yval_n),
        epochs=int(epochs),
        batch_size=int(batch_size),
        callbacks=[ckpt_cb],
        verbose=1
    )

    history4 = dict(hist.history)
    history4["mse_orig"] = ckpt_cb.mse_orig_hist
    history4["val_mse_orig"] = ckpt_cb.val_mse_orig_hist
    return history4

def _concat_histories(hist_list):
    merged = {}
    keys = set()
    for h in hist_list:
        keys.update(h.keys())
    for k in keys:
        merged[k] = []
        for h in hist_list:
            merged[k] += list(h.get(k, []))
    return merged

# -----------------------------
# Evaluation (original units)
# -----------------------------
def evaluate_LTC_power(model, inputs_test, outputs_test):
    Xte = np.asarray(inputs_test, dtype=np.float32)
    Yte = np.asarray(outputs_test, dtype=np.float32).reshape(-1, 1)

    y_scaler = model._y_scaler
    y_pred_n = model.predict(Xte, verbose=0)
    Yhat = y_scaler.inverse(y_pred_n)

    diff = Yhat - Yte
    MAE = float(np.mean(np.abs(diff)))
    MSE = float(np.mean(diff ** 2))
    pct95 = float(np.percentile(np.abs(diff), 95))
    STD = float(np.std(diff))

    print(f"\nMAE: {MAE:.6e}")
    print(f"MSE: {MSE:.6e}")
    print(f"95th Percentile Error: {pct95:.6e}")
    print(f"STD Error: {STD:.6e}")
    return MAE, MSE, pct95, STD, diff

# ============================================================
# PHASE 0) PRETRAINING: Train Ms on Ds; save {theta_l} and {mu_s, sigma_s^2}
# ============================================================
EPOCHS_PRETRAIN = 5
PRETRAIN_VAL_FRAC = 0.1

X_raw_all = np.asarray(_first_if_list(inputs_train_raw), dtype=np.float32)[:int(num_inputRawTrain)]
Y_raw_all = np.asarray(_first_if_list(outputs_train_raw), dtype=np.float32)[:int(num_inputRawTrain)].reshape(-1, 1)

n_raw = len(X_raw_all)
split = max(1, int(round(n_raw * (1 - PRETRAIN_VAL_FRAC))))
Xtr_raw,  Ytr_raw  = X_raw_all[:split],  Y_raw_all[:split]
Xval_raw, Yval_raw = X_raw_all[split:],  Y_raw_all[split:]

# mu_s, sigma_s^2 : normalization adapted on SOURCE train
x_norm_pre = create_normalization_layer(Xtr_raw)

y_scaler_pre = OutputStandardizer(Ytr_raw)

n_features = 5
body_pre = build_LTC_power_body(
    n_features=n_features,
    inter_neurons=32, command_neurons=16, motor_neurons=1,
    head_width=128,
    l2w=0.0, drop=0.0, gaussian_noise_std=0.0,
    ltc_steps=8
)
inp_pre = tf.keras.Input(shape=(n_features,), dtype=tf.float32, name="input")
x_pre = x_norm_pre(inp_pre)
out_pre = body_pre(x_pre)
model_pre = tf.keras.Model(inp_pre, out_pre, name="LTC_power_PRE")
model_pre._y_scaler = y_scaler_pre

pre_ckpt_dir = "./checkpoint_power_pretrain"
os.makedirs(pre_ckpt_dir, exist_ok=True)
pretrain_ckpt_path = f"{pre_ckpt_dir}/pre_train_weights_ltc_power.keras"

history4_pre = train_stage(
    model_pre,
    Xtr_raw, Ytr_raw, Xval_raw, Yval_raw,
    y_scaler=y_scaler_pre,
    lr_init=3.5e-3, decay_rate=0.465133495752869, decay_epochs=65,
    epochs=EPOCHS_PRETRAIN, batch_size=16,
    ckpt_path=pretrain_ckpt_path,
    use_adamw=False, weight_decay=0.0, clipnorm=1.0,
    loss_name="mse", huber_delta=0.01
)

model_pre.load_weights(pretrain_ckpt_path)

print("\n[PHASE 0 - PRE-TRAIN] Eval on REAL TEST (reference):")
evaluate_LTC_power(model_pre, inputs_test, outputs_test)

# ============================================================
# PHASE 1-3) INITIALIZATION -> WARM-UP -> PROGRESSIVE FINE-TUNE
# ============================================================

# ---- Algorithm 2 hyperparameters ----
T_W          = 60          # warm-up epochs
N_STAGES     = 5           # number of progressive unfreezing stages (<= L_TOTAL_LAYERS)
T_TOTAL      = 500         # total epoch budget for the progressive phase (E_N == T_TOTAL)
assert 1 <= N_STAGES <= L_TOTAL_LAYERS, "N must be between 1 and the number of orderable layers."

num_data_array = [1400]
BATCH_FT = 32 * 4

ADAPT_INPUT_NORM_ON_FT = True
ADAPT_Y_SCALER_ON_FT   = True

FT_L2           = 1e-5
FT_DROPOUT      = 0.15
FT_NOISE_STD    = 0.02
FT_USE_ADAMW    = True
FT_WEIGHT_DECAY = 1e-5
FT_USE_HUBER    = True
FT_HUBER_DELTA  = 0.01
FT_LTC_STEPS    = 8

# Per-stage learning rates. Earlier stages (near output, few params
# unfrozen) can tolerate a higher LR; as more/deeper layers unfreeze
# we anneal it down. Adjust to taste.
STAGE_LR_INIT = np.linspace(3.0e-3, 1.5e-3, N_STAGES)  # length N_STAGES, stage k uses STAGE_LR_INIT[k-1]

ft_ckpt_dir = "./checkpoint_power_finetune"
os.makedirs(ft_ckpt_dir, exist_ok=True)

results_MSE = []
results_MAE = []
last_model = None

for num_data in num_data_array:
    print("\n" + "=" * 60)
    print(f"FINE-TUNE on REAL with num_data = {num_data}")
    print("=" * 60)

    mae_iter = []
    mse_iter = []

    for i in range(1):  # REAL splits
        print(f"\n------------------------------")
        print(f"Fine-tune iteration {i} / 5")
        print(f"------------------------------\n")

        # ---------------------------------------------------
        # PHASE 1: INITIALIZATION
        #   Mt <- Ms ;  mu_t, sigma_t^2 <- adapt on Dt
        # ---------------------------------------------------
        model_pre.load_weights(pretrain_ckpt_path)

        Xtr_ft = np.asarray(inputs_train_real[i][0:int(num_data)], dtype=np.float32)
        Ytr_ft = np.asarray(outputs_train_real[i][0:int(num_data)], dtype=np.float32).reshape(-1, 1)

        norm_layer = None
        for layer in model_pre.layers:
            if isinstance(layer, tf.keras.layers.Normalization):
                norm_layer = layer
                break
        assert norm_layer is not None, "Normalization layer not found in pretrain model."

        if ADAPT_INPUT_NORM_ON_FT:
            norm_layer.adapt(Xtr_ft)   # {mu_t, sigma_t^2} <- Dt   (Phase 1, lines 7-8)

        y_scaler_ft = model_pre._y_scaler
        if ADAPT_Y_SCALER_ON_FT:
            y_scaler_ft = OutputStandardizer(Ytr_ft)

        body_ft = build_LTC_power_body(
            n_features=5,
            inter_neurons=32, command_neurons=16, motor_neurons=1,
            head_width=128,
            l2w=FT_L2, drop=FT_DROPOUT, gaussian_noise_std=FT_NOISE_STD,
            ltc_steps=FT_LTC_STEPS
        )

        inp_ft = tf.keras.Input(shape=(5,), dtype=tf.float32, name="input")
        x_ft = norm_layer(inp_ft)
        out_ft = body_ft(x_ft)
        model_ft = tf.keras.Model(inp_ft, out_ft, name=f"LTC_power_FT_iter{i}")
        model_ft._y_scaler = y_scaler_ft

        model_ft.load_weights(pretrain_ckpt_path, by_name=True, skip_mismatch=True)

        stage_histories = []

        # ---------------------------------------------------
        # PHASE 2: WARM-UP
        #   Freeze ALL layers except the output layer;
        #   train theta_L (== unfreeze count 1) for T_w epochs.
        #   (Algorithm 2, lines 9-13: "Freeze LTC; train upper
        #    layers" -- generalized here to "freeze everything
        #    except the output block" so Phase 3 has L-1 layers
        #    left to progressively unfreeze.)
        # ---------------------------------------------------
        freeze_all_ft_layers(model_ft)
        unfreeze_from_output(model_ft, num_unfrozen=1)   # only power_output trainable
        print("[Phase 2 - Warmup] trainable layers:", trainable_report(model_ft))

        warm_ckpt_path = f"{ft_ckpt_dir}/warm_weights_ltc_power_{num_data}_{i}.keras"
        history4_warm = train_stage(
            model_ft,
            Xtr_ft, Ytr_ft,
            inputs_test, outputs_test,
            y_scaler=y_scaler_ft,
            lr_init=3.0e-3,
            decay_rate=0.465133495752869,
            decay_epochs=65,
            epochs=T_W,
            batch_size=BATCH_FT,
            ckpt_path=warm_ckpt_path,
            use_adamw=FT_USE_ADAMW,
            weight_decay=FT_WEIGHT_DECAY,
            clipnorm=1.0,
            loss_name=("huber" if FT_USE_HUBER else "mse"),
            huber_delta=FT_HUBER_DELTA
        )
        stage_histories.append(history4_warm)

        model_ft.load_weights(warm_ckpt_path)

        # ---------------------------------------------------
        # PHASE 3: PROGRESSIVE FINE-TUNING (Algorithm 2, lines 14-26)
        #   E_0 <- T_w
        #   for k = 1..N:
        #       unfreeze theta_{L-k+1} (cumulative)
        #       E_k <- T_w + floor(k*(T_TOTAL - T_w)/N)
        #       train for epochs E_{k-1}+1 .. E_k
        # ---------------------------------------------------
        E_prev = T_W
        final_ckpt_path = f"{ft_ckpt_dir}/final_weights_ltc_power_{num_data}_{i}.keras"

        for k in range(1, N_STAGES + 1):
            unfreeze_from_output(model_ft, num_unfrozen=k)
            E_k = T_W + int(math.floor(k * (T_TOTAL - T_W) / N_STAGES))
            stage_epochs = max(0, E_k - E_prev)

            print(f"\n[Phase 3 - stage k={k}/{N_STAGES}] "
                  f"train epochs {E_prev+1}..{E_k} ({stage_epochs} epochs)")
            print(f"[Phase 3 - stage k={k}] trainable layers:", trainable_report(model_ft))

            stage_ckpt_path = f"{ft_ckpt_dir}/stage{k}_weights_ltc_power_{num_data}_{i}.keras"
            history4_stage = train_stage(
                model_ft,
                Xtr_ft, Ytr_ft,
                inputs_test, outputs_test,
                y_scaler=y_scaler_ft,
                lr_init=float(STAGE_LR_INIT[k - 1]),
                decay_rate=0.465133495752869,
                decay_epochs=65,
                epochs=stage_epochs,
                batch_size=BATCH_FT,
                ckpt_path=stage_ckpt_path,
                use_adamw=FT_USE_ADAMW,
                weight_decay=FT_WEIGHT_DECAY,
                clipnorm=1.0,
                loss_name=("huber" if FT_USE_HUBER else "mse"),
                huber_delta=FT_HUBER_DELTA
            )
            stage_histories.append(history4_stage)

            # Carry forward the best weights found in this stage
            # before moving to the next (more-unfrozen) stage.
            if stage_epochs > 0:
                model_ft.load_weights(stage_ckpt_path)
                model_ft.save_weights(final_ckpt_path)

            E_prev = E_k

        history4 = _concat_histories(stage_histories)

        np.savez(
            f"{ft_ckpt_dir}/history4_power_ndata{num_data}_iter{i}.npz",
            **{k_: np.asarray(v_) for k_, v_ in history4.items()}
        )

        # Evaluate best FT weights (best across all Phase-3 stages)
        model_ft.load_weights(final_ckpt_path)
        last_model = model_ft
        MAE, MSE, pct95, STD, diff = evaluate_LTC_power(model_ft, inputs_test, outputs_test)
        mae_iter.append(MAE)
        mse_iter.append(MSE)

        print(f"\n[iter {i}] done. Best checkpoint MSE shown above.\n")

    print(f"\nBest MAE for {num_data}: {min(mae_iter)}")
    print(f"Best MSE for {num_data}: {min(mse_iter)}")

    results_MAE.append(mae_iter)
    results_MSE.append(mse_iter)

# ============================================================
# EXPORT CSV (last model)
# ============================================================
def save_csv(name, arr):
    np.savetxt(name, np.asarray(arr), delimiter=",", fmt="%.10g")

if last_model is not None:
    y_pred = last_model._y_scaler.inverse(last_model.predict(np.asarray(inputs_test, dtype=np.float32), verbose=0))
    save_csv("Input_Power_TL.csv", inputs_test)
    save_csv("Output_Power_TL.csv", outputs_test)
    save_csv("Prediction_Power_TL.csv", y_pred)
    print("\nExport complete: CSV files saved (TL).")